In [3]:
import os
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
from tqdm import tqdm
# from extract_features import GenerateFeatures
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from xgboost import XGBClassifier, plot_importance
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import scale, StandardScaler
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

In [4]:
train_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/extracted_features_data/3_class/train_set/two_class_4s_0.8.csv'
test_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/extracted_features_data/3_class/test_set/two_class_4s_0.8.csv'

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)


train_data.drop(['center_time', 'start_time', 'end_time'], axis= 1, inplace=True)
test_data.drop(['center_time', 'start_time', 'end_time'], axis= 1, inplace=True)

X_train = train_data.drop(columns=['label', 'experiment_id'])
y_train = train_data['label']
groups_train = train_data['experiment_id']

X_test = test_data.drop(columns=['label', 'experiment_id'])
y_test = test_data['label']
groups_test = test_data['experiment_id']

In [4]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

In [5]:
model = DecisionTreeClassifier()

In [ ]:
def cross_validation_hyperparameter_tuning(X, y, groups, model_type, random_state=42):
    """
    Tune hyperparameters with 5 fold cross-validation
    """
    
    results = {}
    

In [1]:
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
import optuna

/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Cross validation splitter
sgkf = StratifiedGroupKFold(n_splits=5, random_state=42)

In [ ]:
# Track scores
best_configs = []

In [ ]:
def define_search_space(self, trial):
        """Define model hyperparameter search space."""
        model_name = trial.suggest_categorical('model', ['rf', 'xgb'])
        imbalance_technique = trial.suggest_categorical(
            'imbalance_technique', ['none', 'smote', 'smote_tomek', 'tomek_links']
        )
        
        if model_name == 'dt':
            params = {
                'model': 'dt',
                'max_depth': trial.suggest_int('max_depth', 0, 20),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            }

        elif model_name == 'rf':
            params = {
                'model': 'rf',
                'n_estimators': trial.suggest_int('n_estimators', 50, 500),
                'max_depth': trial.suggest_int('max_depth', 0, 20),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
                'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
            }
        elif model_name == 'xgb':  
            params = {
                'model': 'xgb',
                'n_estimators': trial.suggest_int('n_estimators', 50, 500),
                'max_depth': trial.suggest_int('max_depth', 0, 20),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'gamma': trial.suggest_float('gamma', 0, 5),
                'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
                'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
                'early_stopping_rounds': trial.suggest_int('early_stopping_rounds', 10, 100),
            }
        return params

In [ ]:
def create_model(self, params):
        """Instantiate the model with given parameters."""
        if params['model'] == 'dt':
            return DecisionTreeClassifier(
                max_depth=params['max_depth'],
                min_samples_split=params['min_samples_split'],
                min_samples_leaf=params['min_samples_leaf'],
                max_features=params['max_features'],
                random_state=self.random_state
            )
        if params['model'] == 'rf':
            return RandomForestClassifier(
                n_estimators=params['n_estimators'],
                max_depth=params['max_depth'],
                min_samples_split=params['min_samples_split'],
                min_samples_leaf=params['min_samples_leaf'],
                max_features=params['max_features'],
                bootstrap=params['bootstrap'],
                random_state=self.random_state,
                n_jobs=-1
            )
        elif params['model'] == 'xgb':
            return XGBClassifier(
                n_estimators=params['n_estimators'],
                max_depth=params['max_depth'],
                learning_rate=params['learning_rate'],
                subsample=params['subsample'],
                colsample_bytree=params['colsample_bytree'],
                min_child_weight=params['min_child_weight'],
                gamma=params['gamma'],
                reg_alpha=params['reg_alpha'],
                reg_lambda=params['reg_lambda'],
                random_state=self.random_state,
                n_jobs=-1,
                eval_metric='logloss',
                early_stopping_rounds=params['early_stopping_rounds'],
                verbosity=0,
                use_label_encoder=False
            )

In [ ]:
def tune_hyperparams(trial, X_train, y_train, groups_train):
        """Inner CV loop for hyperparameter tuning."""
        params = define_search_space(trial)
        fold_scores = []

        for train_idx, test_idx in sgkf.split(X_train, y_train, groups_train):
            X_train_fold = X_train.iloc[train_idx]
            X_test_fold = X_train.iloc[test_idx]
            y_train_fold = y_train.iloc[train_idx]
            y_test_fold = y_train.iloc[test_idx]


            model = create_model(params)

            if params['model'] == 'xgb':
                model.fit(
                    X_train_fold,
                    y_train_fold,
                    eval_set=[(X_test_fold, y_test_fold)],
                    verbose=False
                )
            else:
                model.fit(X_train_fold, y_train_fold)

            y_pred = model.predict(X_test_fold)
            score = f1_score(y_test_fold, y_pred, zero_division=0)
            fold_scores.append(score)

In [ ]:
def run_nested_cv():
    """Run the full nested CV optimization."""
    print("Starting Nested Cross-Validation...")

    for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X_train, y_train_encoded, groups_train)):
        print(f"\n=== Fold {fold_idx + 1} ===")

        X_train_fold = X_train.iloc[train_idx]
        X_test_fold = X_train.iloc[test_idx]
        y_train_fold = pd.Series(y_train[train_idx])
        y_test_fold = pd.Series(y_train[test_idx])
        groups_train_fold = groups_train.iloc[train_idx]

        # Inner CV hyperparameter tuning
        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=42),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)
        )
        study.optimize(
            lambda trial: inner_cv_objective(trial, X_train, y_train, groups_train),
            n_trials=n_trials,
            show_progress_bar=True
        )

        best_params = study.best_params
        print(f"Best parameters: {best_params}")

        # Retrain final model on full outer train set
        X_train_balanced, y_train_balanced = apply_imbalance_handling(
        X_train_outer, y_train_outer, best_params['imbalance_technique']
        )
        final_model = create_model(best_params)

        if best_params['model'] == 'xgb':
            final_model.fit(
                X_train_balanced,
                y_train_balanced,
                eval_set=[(X_test_outer, y_test_outer)],
                verbose=False
            )
        else:
            final_model.fit(X_train_balanced, y_train_balanced)

        # Evaluate on outer test set
        y_pred = final_model.predict(X_test_outer)
        fold_metrics = calculate_metrics(y_test_outer, y_pred)
        
        print(f"Outer fold metrics: {fold_metrics}")
        outer_scores.append(fold_metrics)
        best_configs.append(best_params)

        return _summarize_results()